# Chapter 14 — Change One Thing

**Book alignment:** DSPy From First Principles, Chapter 14

**Question this notebook isolates:** Does changing only the objective — holding corpus, program, model, and budgets fixed — change what search rewards?

In [ ]:
from pathlib import Path
import random
import sys

random.seed(13)


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "experiments" / "dspy-from-first-principles" / "common").exists():
            return candidate
    raise RuntimeError(
        "Run this notebook from a checkout containing experiments/dspy-from-first-principles"
    )


REPO_ROOT = find_repo_root(Path.cwd().resolve())
EXP_ROOT = REPO_ROOT / "experiments" / "dspy-from-first-principles"
sys.path.insert(0, str(EXP_ROOT))

import dspy  # imported for construction only; no LM is ever called

from common.data import canonical_split
from common.metrics import editorial_metric

## The same deletion under two rulers

Corpus, program, model, and budgets are held fixed; only the ruler moves. Under v1 the ed-035 deletion is a rounding error. Under a deterministic constraint objective it is a named failure — `qualifier:regularly` — and correct restraint stops being penalised.

In [ ]:
split = canonical_split()
dev_by_id = {c.case_id: c for c in split.dev}
ed035 = dev_by_id["ed-035"]
good = ed035.reference_rewrite
bad = good.replace("regularly ", "")


def qualifier_present(candidate: str, qualifier: str = "regularly") -> bool:
    import re
    return re.search(rf"\b{re.escape(qualifier)}\b", candidate.lower()) is not None


def constraint_score(passes_named: bool, unchanged_legitimate: bool, changed: bool) -> float:
    """Deterministic replay of the chapter's rule: named failures fail loudly;
    legitimate restraint earns full credit instead of the 0.65 cap."""
    if not passes_named:
        return 0.0  # named failure: qualifier dropped
    if unchanged_legitimate and not changed:
        return 1.0  # restraint is a valid answer here
    return 1.0


v1_good = editorial_metric(ed035, good, version="v1")
v1_bad = editorial_metric(ed035, bad, version="v1")
named_good = qualifier_present(good)
named_bad = qualifier_present(bad)

restraint_rows = []
for case_id in ("ed-034", "ed-037"):
    case = dev_by_id[case_id]
    v1 = editorial_metric(case, case.sentence, version="v1")
    restraint_rows.append({
        "case_id": case_id,
        "v1_unchanged": round(v1.score, 4),
        "constraint_unchanged": constraint_score(True, True, v1.changed_score == 1.0),
    })

({
    "v1_good": round(v1_good.score, 4),
    "v1_bad": round(v1_bad.score, 4),
    "named_good": named_good,
    "named_bad": named_bad,
    "restraint": restraint_rows,
})

In [ ]:
assert abs((v1_good.score - v1_bad.score) - 1 / 30) < 1e-9  # v1: a 0.033 rounding error
assert named_good is True and named_bad is False  # constraint: a named failure
assert constraint_score(named_bad, False, True) == 0.0
for r in restraint_rows:
    assert abs(r["v1_unchanged"] - 0.65) < 1e-9  # v1 caps restraint ...
    assert r["constraint_unchanged"] == 1.0  # ... the constraint objective does not

print("v1 prices the deletion at 0.033; the constraint ruler names it qualifier:regularly")
print("restraint floor gone: 0.65 under v1 becomes full credit where leaving text alone is correct")

## Deterministic is not correct

Reproducibility is not trustworthiness. The chapter's audit found forbidden terms like `"\\n"` that strip to the empty string — and the empty string is a substring of everything, so a naive check fails every candidate. Demonstrate the bug and the fix on a real dev case.

In [ ]:
ed016 = {c.case_id: c for c in split.train}["ed-016"]
assert any(t.strip() == "" for t in ed016.forbidden_terms)  # e.g. "\n"


def buggy_forbidden_hit(candidate: str, terms) -> bool:
    return any(t.strip().lower() in candidate.lower() for t in terms)


def fixed_forbidden_hit(candidate: str, terms) -> bool:
    return any(t.strip() and t.strip().lower() in candidate.lower() for t in terms)


reference = ed016.reference_rewrite
({
    "forbidden_terms": list(ed016.forbidden_terms),
    "buggy_flags_reference": buggy_forbidden_hit(reference, ed016.forbidden_terms),
    "fixed_flags_reference": fixed_forbidden_hit(reference, ed016.forbidden_terms),
})

In [ ]:
assert buggy_forbidden_hit(reference, ed016.forbidden_terms) is True  # stable, reproducible, wrong
assert fixed_forbidden_hit(reference, ed016.forbidden_terms) is False
assert "" in "anything at all"  # the mechanism: empty matches everything

print("the buggy check fails a known-good rewrite the same way every time")
print("determinism buys inspectability; twelve unit tests buy trust")

## What we earned

One controlled change — the ruler, and nothing else — moved the outcome: the ed-035 deletion became a selectable-against named failure, the restraint floor disappeared, and the deterministic instrument itself had to be debugged before it deserved trust. DSPy optimization is objective-sensitive and optimizer-dependent: named feedback unlocked the reflective loop while leaving demonstration-only and bounded scalar searches flat or null.

Notebook 15 / Chapter 15 moves to the regime the final prediction needs: context, retrieval, and memory, where failures arrive with test names and tracebacks instead of blended scalars.